In [1]:
import pandas as pd

df = pd.read_csv('data/P1.csv')

df.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,P1,08/08/2025,20:15,Casa Pia,Sp Lisbon,0,2,A,0,1,...,1.93,1.93,1.90,2.00,1.93,1.95,1.82,1.92,1.91,2.04
1,P1,09/08/2025,15:30,Nacional,Gil Vicente,0,2,A,0,2,...,1.80,2.05,1.89,2.02,1.83,2.05,1.78,1.97,1.87,2.12
2,P1,09/08/2025,20:30,Arouca,AVS,3,1,H,1,0,...,1.98,1.88,1.97,1.93,1.98,1.90,1.90,1.79,2.02,1.95
3,P1,10/08/2025,17:00,Famalicao,Santa Clara,3,0,H,2,0,...,1.93,1.93,2.00,1.90,1.93,1.93,1.78,1.89,1.93,2.05
4,P1,10/08/2025,20:30,Moreirense,Alverca,2,1,H,1,1,...,1.95,1.90,2.05,1.87,1.95,1.90,1.88,1.84,2.05,1.94


In [2]:

print("Número de colunas originais:", len(df.columns))

colunas_importantes = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR']


df_limpo = df[colunas_importantes].copy()


df_limpo.head()

Número de colunas originais: 131


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR
0,08/08/2025,Casa Pia,Sp Lisbon,0,2,A
1,09/08/2025,Nacional,Gil Vicente,0,2,A
2,09/08/2025,Arouca,AVS,3,1,H
3,10/08/2025,Famalicao,Santa Clara,3,0,H
4,10/08/2025,Moreirense,Alverca,2,1,H


In [3]:
import pandas as pd 


df_limpo['Date'] = pd.to_datetime(df_limpo['Date'], dayfirst=True)
df_limpo = df_limpo.sort_values(by='Date')


def calcula_pontos_casa(resultado):
    if resultado == 'H': 
        return 3
    elif resultado == 'D': 
        return 1
    else: 
        return 0


def calcula_pontos_fora(resultado):
    if resultado == 'A': 
        return 3
    elif resultado == 'D': 
        return 1
    else: 
        return 0


df_limpo['HomePoints'] = df_limpo['FTR'].apply(calcula_pontos_casa)
df_limpo['AwayPoints'] = df_limpo['FTR'].apply(calcula_pontos_fora)


df_limpo.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HomePoints,AwayPoints
0,2025-08-08,Casa Pia,Sp Lisbon,0,2,A,0,3
1,2025-08-09,Nacional,Gil Vicente,0,2,A,0,3
2,2025-08-09,Arouca,AVS,3,1,H,3,0
3,2025-08-10,Famalicao,Santa Clara,3,0,H,3,0
4,2025-08-10,Moreirense,Alverca,2,1,H,3,0


In [4]:

df_limpo['HomeForm'] = 0
df_limpo['AwayForm'] = 0


pontos_totais = {}


for index, row in df_limpo.iterrows():
    casa = row['HomeTeam']
    fora = row['AwayTeam']
    
    
    if casa not in pontos_totais:
        pontos_totais[casa] = 0
    if fora not in pontos_totais:
        pontos_totais[fora] = 0
        
    
    df_limpo.at[index, 'HomeForm'] = pontos_totais[casa]
    df_limpo.at[index, 'AwayForm'] = pontos_totais[fora]
    
    
    pontos_totais[casa] += row['HomePoints']
    pontos_totais[fora] += row['AwayPoints']


df_limpo.head(15)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HomePoints,AwayPoints,HomeForm,AwayForm
0,2025-08-08,Casa Pia,Sp Lisbon,0,2,A,0,3,0,0
1,2025-08-09,Nacional,Gil Vicente,0,2,A,0,3,0,0
2,2025-08-09,Arouca,AVS,3,1,H,3,0,0,0
3,2025-08-10,Famalicao,Santa Clara,3,0,H,3,0,0,0
4,2025-08-10,Moreirense,Alverca,2,1,H,3,0,0,0
5,2025-08-10,Sp Braga,Tondela,3,0,H,3,0,0,0
6,2025-08-11,Estoril,Estrela,1,1,D,1,1,0,0
7,2025-08-11,Porto,Guimaraes,3,0,H,3,0,0,0
8,2025-08-15,AVS,Casa Pia,0,2,A,0,3,0,0
11,2025-08-16,Estrela,Benfica,0,1,A,0,3,1,0


In [5]:
# 1. Definir o nosso Alvo (O que queremos prever)
y = df_limpo['FTR']

# 2. Definir as nossas Variáveis (O que vamos usar para prever)
# Vamos apagar a Data e as colunas que são "batota" (Data Leakage)
X = df_limpo.drop(columns=['Date', 'FTHG', 'FTAG', 'FTR', 'HomePoints', 'AwayPoints'])

# 3. Converter o texto (Nomes das Equipas) em números (0 e 1)
X = pd.get_dummies(X, columns=['HomeTeam', 'AwayTeam'], dtype=int)

# Vamos ver a nossa tabela X final, pronta para ir para o forno!
X.head()

,HomeForm,AwayForm,HomeTeam_AVS,HomeTeam_Alverca,HomeTeam_Arouca,HomeTeam_Benfica,HomeTeam_Casa Pia,HomeTeam_Estoril,HomeTeam_Estrela,HomeTeam_Famalicao,...,AwayTeam_Gil Vicente,AwayTeam_Guimaraes,AwayTeam_Moreirense,AwayTeam_Nacional,AwayTeam_Porto,AwayTeam_Rio Ave,AwayTeam_Santa Clara,AwayTeam_Sp Braga,AwayTeam_Sp Lisbon,AwayTeam_Tondela
0,0,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split (X, y, test_size=0.2, random_state=42)

print(f"Jogos para o modelo estudar: {len(X_train)}")
print(f"Jogos para o modelo testar: {len(X_test)}")

modelo=RandomForestClassifier(n_estimators=200, random_state=42)

modelo.fit(X_train, y_train)

previsoes= modelo.predict(X_test)

precisao = accuracy_score(y_test, previsoes)
print(f"\n--->Precisão do modelo: {precisao * 100:.2f}%")

Jogos para o modelo estudar: 237
Jogos para o modelo testar: 60

--->Precisão do modelo: 53.33%


In [7]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# 1. O Tradutor: Converter 'H', 'D', 'A' em 0, 1, 2 para o XGBoost não se queixar
tradutor = LabelEncoder()
y_train_numeros = tradutor.fit_transform(y_train)
y_test_numeros = tradutor.transform(y_test)

# 2. Criar o Monstro (Vamos usar as mesmas 200 "árvores" para ser justo)
# A learning_rate (taxa de aprendizagem) impede o modelo de ser demasiado confiante e errar
modelo_xgb = XGBClassifier(
    n_estimators=200, 
    learning_rate=0.05, 
    max_depth=3, # Árvores baixinhas para evitar que ele decore o ficheiro
    random_state=42
)

# 3. Treinar a máquina
print("A treinar o XGBoost. Aguenta coração...")
modelo_xgb.fit(X_train, y_train_numeros)

# 4. Fazer o teste final e ver o resultado
previsoes_xgb = modelo_xgb.predict(X_test)
precisao_xgb = accuracy_score(y_test_numeros, previsoes_xgb)

print(f"\n🚀 Precisão Final do XGBoost: {precisao_xgb * 100:.2f}%")

A treinar o XGBoost. Aguenta coração...

🚀 Precisão Final do XGBoost: 40.00%


In [8]:
import joblib

# 1. Agarrar no nosso modelo vencedor das 200 árvores (da Célula 6)
modelo_final = modelo 

# 2. Exportar o modelo para um ficheiro chamado .pkl (Pickle)
joblib.dump(modelo_final, 'modelo_primeira_liga.pkl')

# 3. Guardar a "lista de ingredientes" (as colunas do nosso X)
colunas_do_modelo = list(X.columns)
joblib.dump(colunas_do_modelo, 'colunas_modelo.pkl')

print("🧠 Cérebro do modelo e colunas guardados com sucesso no teu Mac!")

🧠 Cérebro do modelo e colunas guardados com sucesso no teu Mac!


In [9]:
import joblib
import pandas as pd

# 1. Acordar o "Cérebro" (Carregar ficheiros)
modelo = joblib.load('modelo_primeira_liga.pkl')
colunas = joblib.load('colunas_modelo.pkl')

# 2. Criar uma tabela em branco com o formato exato que o modelo exige
jogo_novo = pd.DataFrame(0, index=[0], columns=colunas)

# 3. Preencher os dados do nosso jogo fictício
# IMPORTANTE: Os nomes das equipas têm de estar escritos exatamente como no CSV
jogo_novo.at[0, 'HomeForm'] = 15  # Pontos do Porto à entrada para o jogo
jogo_novo.at[0, 'AwayForm'] = 14  # Pontos do Benfica à entrada para o jogo
jogo_novo.at[0, 'HomeTeam_Porto'] = 1
jogo_novo.at[0, 'AwayTeam_Benfica'] = 1

# 4. Pedir à máquina para prever!
probabilidades = modelo.predict_proba(jogo_novo)[0]

print("⚽ PREVISÃO DO GRANDE CLÁSSICO ⚽")
print(f"Vitória do Porto (Casa): {probabilidades[2] * 100:.1f}%")
print(f"Empate:                  {probabilidades[1] * 100:.1f}%")
print(f"Vitória do Benfica (Fora): {probabilidades[0] * 100:.1f}%")

⚽ PREVISÃO DO GRANDE CLÁSSICO ⚽
Vitória do Porto (Casa): 28.5%
Empate:                  49.5%
Vitória do Benfica (Fora): 22.0%
